# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through exploring the FAIR<sup>2</sup> protein abundance and modification dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading

Load metadata and explore available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic dataset information
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview

Review available record sets, their fields, and associated `@id` values.

In [ ]:
from pprint import pprint

# List all record sets by their @id
print("Available record sets (by @id):")
record_set_ids = []
for rset in dataset.metadata.recordSet:
    print(f"  - {rset['@id']}: {rset.get('name', '')}")
    record_set_ids.append(rset["@id"])

# For each record set, print their fields and columns by @id
for rset in dataset.metadata.recordSet:
    print(f"\n=== Record Set @id: {rset['@id']} - {rset.get('name', '')} ===")
    if "field" in rset and rset["field"]:
        print("  Fields:")
        for field in rset["field"]:
            print(f"    - {field['@id']}: {field.get('name', field['@id'])} [type: {field.get('dataType', '-')}]" )
            if 'column' in field and isinstance(field['column'], dict):
                print(f"      column @id: {field['column'].get('@id','')}")
            elif 'column' in field and isinstance(field['column'], list):
                for col in field['column']:
                    print(f"      column @id: {col.get('@id','')}")
    else:
        print("  No fields defined.")

## 3. Data Extraction

Load data from specific record sets into pandas DataFrames for analysis. Use the `@id` values found above.

The main data table is typically located in the first record set. We demonstrate loading all available record sets for completeness.

In [ ]:
# Set up the list of record set @id values found above
record_sets = record_set_ids
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for RecordSet {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    # Only load if records are non-empty
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records. Columns:", df.columns.tolist())
    else:
        print(f"No records found in record set {record_set_id}.")

# Pick the main record set for analysis (usually the first or the one with most records)
main_record_set_id = None
max_rows = 0
for k, v in dataframes.items():
    if len(v) > max_rows:
        max_rows = len(v)
        main_record_set_id = k

if main_record_set_id:
    print(f"\nMain record set selected for analysis: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No records found in any record set.")

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing: filter, normalize, and group by fields of interest, using column/field `@id` where applicable.

*Example below selects the numeric field 'cr:coverage' (`@id` shown), filters for values above 10, normalizes, and groups by 'cr:accession' (protein identifier).*

In [ ]:
main_df = dataframes[main_record_set_id]

# Identify a numeric field to analyze
# The most common ones are typically 'cr:coverage', 'cr:peptide_count', or similar
numeric_field_id = None
for col in main_df.columns:
    if col.lower().find('coverage') >= 0 or col.lower().find('count') >= 0 or col.lower().find('mw') >= 0:
        numeric_field_id = col
        break

if numeric_field_id is None:
    # fall back to first numeric field
    possible_numeric = main_df.select_dtypes('number').columns
    if len(possible_numeric):
        numeric_field_id = possible_numeric[0]
    else:
        print("Could not find numeric field for EDA.")
        numeric_field_id = main_df.columns[0]

print(f"Using numeric field '@id': {numeric_field_id}")

threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
try:
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
except Exception as e:
    print(f"Error filtering field '{numeric_field_id}': {e}")
    filtered_df = main_df.copy()

print(f"Filtered records where {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize numeric field
if pd.api.types.is_numeric_dtype(filtered_df[numeric_field_id]):
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized '{numeric_field_id}' for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print(f"Cannot normalize non-numeric field: {numeric_field_id}")

# Group by another field if available (e.g. 'cr:accession', 'cr:description', etc.)
group_field = None
for col in filtered_df.columns:
    if col.lower().find('accession') >= 0:
        group_field = col
        break
if group_field is None:
    group_field = filtered_df.columns[0]  # Just group by the first field

if group_field in filtered_df.columns:
    print(f"\nGrouped mean of numeric fields by '{group_field}':")
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
    display(grouped_df.head())

## 5. Visualization

Visualize the distribution of the selected numeric field and relationships with other fields, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(main_df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel(numeric_field_id)
plt.title(f"Distribution of {numeric_field_id}")
plt.show()

# If normalized field exists, draw violinplot by group (may look odd for accession, but illustrative)
if f"{numeric_field_id}_normalized" in filtered_df.columns and group_field in filtered_df.columns:
    subdf = filtered_df[[group_field, f"{numeric_field_id}_normalized"]].dropna()
    # Plot just top 10 groups by size
    topn = subdf[group_field].value_counts().index[:10]
    plt.figure(figsize=(10,6))
    sns.violinplot(data=subdf[subdf[group_field].isin(topn)], x=group_field, y=f"{numeric_field_id}_normalized")
    plt.title(f"Normalized {numeric_field_id} by {group_field} (top 10)")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

- We loaded and explored the FAIR<sup>2</sup> dataset on protein abundance and modification using `mlcroissant`.
- Main record set and fields were identified by their `@id` for transparent referencing.
- Sample EDA and visualizations were performed, including basic grouping and normalization of numeric fields.

Further downstream analysis can now be performed using the structured DataFrames, leveraging clear schema references for reproducibility.